<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026-10-18-EP008-tetris-ML-SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 서버 연결 실패가 풀리지 않음

1. 구글 collob에 SVM 머신러닝 모델을 이용한 테트리스 게임을 AI가 하는 것 만들기
2. data는 유명한 tetris 알고리즘에서 합성데이터를 코드로 생성하기
3. SVM 머신러닝으로 학습
4.별도의 로직이 전혀 없는  html gui가 websocket으로 머신러닝 로직과 연결되어 게임할 수 있게 진행
5. Cell이 길어질 수 있으므로, Cell에 번호를 붙이고 순서대로 코드 보여줄 것


---



In [1]:
# Cell 1: 패키지 설치 및 비동기 환경 설정
!pip install nest-asyncio websockets scikit-learn numpy

import nest_asyncio
nest_asyncio.apply()  # Colab의 이벤트 루프 충돌 방지
print("설치 및 기본 설정 완료!")


설치 및 기본 설정 완료!


In [2]:
# Cell 2: 테트리스 평가 피처 기반 합성 데이터 생성 및 SVM 모델 학습
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

def generate_tetris_synthetic_data(num_samples=2000):
    np.random.seed(42)
    holes = np.random.randint(0, 8, size=num_samples)
    agg_height = np.random.randint(0, 20, size=num_samples)
    bumpiness = np.random.randint(0, 15, size=num_samples)
    lines_cleared = np.random.randint(0, 5, size=num_samples)

    X = np.column_stack((holes, agg_height, bumpiness, lines_cleared))
    score = (holes * -7.5) + (agg_height * -4.5) + (bumpiness * -3.5) + (lines_cleared * 5.0)
    median_score = np.median(score)
    y = (score >= median_score).astype(int)
    return X, y

X, y = generate_tetris_synthetic_data()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("SVM 모델 학습 시작...")
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train, y_train)

accuracy = svm_model.score(X_test, y_test)
print(f"SVM 모델 학습 완료! 테스트 정확도: {accuracy * 100:.2f}%")

joblib.dump(svm_model, 'svm_tetris_model.pkl')
joblib.dump(scaler, 'tetris_scaler.pkl')


SVM 모델 학습 시작...
SVM 모델 학습 완료! 테스트 정확도: 97.00%


['tetris_scaler.pkl']

In [3]:
# Cell 3: WebSocket AI 연동 서버 정의 및 실행
import asyncio
import json
import websockets
import numpy as np
import joblib
from google.colab import output

# 모델 및 스케일러 로드
model = joblib.load('svm_tetris_model.pkl')
scaler = joblib.load('tetris_scaler.pkl')

SHAPES = {
    'I': [[0,0,0,0],[1,1,1,1],[0,0,0,0],[0,0,0,0]],
    'O': [[1,1],[1,1]],
    'T': [[0,1,0],[1,1,1],[0,0,0]],
    'S': [[0,1,1],[1,1,0],[0,0,0]],
    'Z': [[1,1,0],[0,1,1],[0,0,0]],
    'J': [[1,0,0],[1,1,1],[0,0,0]],
    'L': [[0,0,1],[1,1,1],[0,0,0]]
}

def rotate_matrix(matrix):
    return [list(x) for x in zip(*matrix[::-1])]

def get_heuristics(grid):
    grid = np.array(grid)
    R, C = grid.shape

    heights = []
    for col in range(C):
        has_block = np.where(grid[:, col] > 0)[0]
        heights.append(R - has_block[0] if len(has_block) > 0 else 0)

    agg_height = sum(heights)
    bumpiness = sum(abs(heights[i] - heights[i+1]) for i in range(C - 1))

    holes = 0
    for col in range(C):
        block_found = False
        for row in range(R):
            if grid[row, col] > 0:
                block_found = True
            elif block_found and grid[row, col] == 0:
                holes += 1

    lines_cleared = sum(1 for row in grid if all(cell > 0 for cell in row))
    return [holes, agg_height, bumpiness, lines_cleared]

def simulate_and_find_best(grid, block_type):
    best_x = 3
    best_rot = 0
    max_prob = -1.0

    original_shape = SHAPES.get(block_type, SHAPES['I'])
    grid_arr = np.array(grid)
    R, C = grid_arr.shape

    shape = [row[:] for row in original_shape]

    for r_idx in range(4):
        s_w = len(shape[0])
        s_h = len(shape)

        for x in range(0, C - s_w + 1):
            target_y = 0
            for y in range(0, R - s_h + 1):
                collision = False
                for r in range(s_h):
                    for c in range(s_w):
                        if shape[r][c] > 0:
                            if y + r >= R or x + c >= C or grid_arr[y + r][x + c] > 0:
                                collision = True
                                break
                    if collision: break
                if collision:
                    target_y = y - 1
                    break
                target_y = y

            if target_y < 0: continue

            temp_grid = grid_arr.copy()
            for r in range(s_h):
                for c in range(s_w):
                    if shape[r][c] > 0 and (target_y + r) < R and (x + c) < C:
                        temp_grid[target_y + r][x + c] = 1

            features = get_heuristics(temp_grid)
            features_scaled = scaler.transform([features])

            # [버그 수정 완료] 클래스 1(좋은 수)에 대칭되는 인덱스 스칼라 값 정확히 추출
            prob = model.predict_proba(features_scaled)[0][1]

            if prob > max_prob:
                max_prob = prob
                best_x = x
                best_rot = r_idx

        shape = rotate_matrix(shape)

    return best_x, best_rot

async def handler(websocket):
    print("HTML GUI 프론트엔드가 연결되었습니다.")
    try:
        async for message in websocket:
            data = json.loads(message)
            grid = data['grid']
            current_piece = data['currentPiece']

            # 예측 연산 중 예외 상황 전면 방어
            try:
                target_x, target_rot = simulate_and_find_best(grid, current_piece)
            except Exception as calc_err:
                print(f"AI 연산 에러 발생 (기본값 대체): {calc_err}")
                target_x, target_rot = 3, 0

            await websocket.send(json.dumps({'targetX': int(target_x), 'targetRot': int(target_rot)}))
    except websockets.ConnectionClosed:
        print("프론트엔드 연결 종료")

async def start_server():
    # 혼선 방지를 위해 포트를 8999로 전면 수정
    async with websockets.serve(handler, "0.0.0.0", 8999):
        print("WebSocket Server가 포트 8999에서 안전하게 대기 중...")
        await asyncio.Future()

# 보안 WSS 프록시 링크 생성
http_proxy_url = output.eval_js("google.colab.kernel.proxyPort(8999)")
websocket_wss_url = http_proxy_url.replace("http", "ws")
print(f"\n[성공] 연동용 프록시 웹소켓 주소:")
print(f"👉 {websocket_wss_url}\n")

asyncio.create_task(start_server())



[성공] 연동용 프록시 웹소켓 주소:
👉 wss://8999-gpu-t4-s-kkb-euw4c2-2angmoj8jvpwy-c.europe-west4-2.prod.colab.dev



<Task pending name='Task-1' coro=<start_server() running at /tmp/ipykernel_1249/1495899573.py:123>>

In [4]:
# Cell 4: 코랩용 프록시 주소를 주입받은 HTML GUI 출력
from IPython.display import HTML, display

html_code = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>SVM Tetris AI GUI</title>
    <style>
        body {{ background: #222; color: #fff; font-family: sans-serif; text-align: center; }}
        #game-container {{ display: flex; justify-content: center; margin-top: 20px; }}
        canvas {{ border: 2px solid #555; background: #111; }}
        #info {{ margin-left: 20px; text-align: left; }}
        .status {{ color: #0f0; font-weight: bold; }}
    </style>
</head>
<body>
    <h1>SVM Tetris AI Player</h1>
    <p>WebSocket 연동 상태: <span id="ws-status" style="color: yellow;">연결 시도 중...</span></p>

    <div id="game-container">
        <canvas id="tetris" width="240" height="480"></canvas>
        <div id="info">
            <h2>Score: <span id="score">0</span></h2>
            <h3>Next: <span id="next-piece">-</span></h3>
            <p>서버 연동을 통해 별도 조작 없이<br>SVM 모델이 게임을 플레이합니다.</p>
        </div>
    </div>

    <script>
        const canvas = document.getElementById('tetris');
        const context = canvas.getContext('2d');
        context.scale(24, 24);

        const COLS = 10;
        const ROWS = 20;
        let score = 0;
        const grid = Array.from({{length: ROWS}}, () => Array(COLS).fill(0));

        const SHAPES = {{
            'I': [[0,0,0,0],[1,1,1,1],[0,0,0,0],[0,0,0,0]],
            'O': [[1,1],[1,1]],
            'T': [[0,1,0],[1,1,1],[0,0,0]],
            'S': [[0,1,1],[1,1,0],[0,0,0]],
            'Z': [[1,1,0],[0,1,1],[0,0,0]],
            'J': [[1,0,0],[1,1,1],[0,0,0]],
            'L': [[0,0,1],[1,1,1],[0,0,0]]
        }};
        const COLORS = {{ 'I': 'cyan', 'O': 'yellow', 'T': 'purple', 'S': 'green', 'Z': 'red', 'J': 'blue', 'L': 'orange' }};
        const PIECES = ['I', 'O', 'T', 'S', 'Z', 'J', 'L'];

        let currentPiece = {{ shape: [], type: '', x: 0, y: 0 }};
        let nextPieceType = PIECES[Math.floor(Math.random() * PIECES.length)];

        function resetPiece() {{
            currentPiece.type = nextPieceType;
            currentPiece.shape = SHAPES[currentPiece.type].map(row => [...row]);
            currentPiece.x = Math.floor((COLS - currentPiece.shape[0].length) / 2);
            currentPiece.y = 0;
            nextPieceType = PIECES[Math.floor(Math.random() * PIECES.length)];
            document.getElementById('next-piece').innerText = nextPieceType;

            if (checkCollision(grid, currentPiece)) {{
                grid.forEach(row => row.fill(0));
                score = 0;
                document.getElementById('score').innerText = score;
            }}
            requestAIMove();
        }}

        function checkCollision(g, p) {{
            for (let r = 0; r < p.shape.length; r++) {{
                for (let c = 0; c < p.shape[r].length; c++) {{
                    if (p.shape[r][c]) {{
                        let nextY = p.y + r;
                        let nextX = p.x + c;
                        if (nextY >= ROWS || nextX < 0 || nextX >= COLS || (nextY >= 0 && g[nextY][nextX])) {{
                            return true;
                        }}
                    }}
                }}
            }}
            return false;
        }}

        function merge(g, p) {{
            p.shape.forEach((row, r) => {{
                row.forEach((value, c) => {{
                    if (value && (p.y + r) >= 0) g[p.y + r][p.x + c] = p.type;
                }});
            }});
        }}

        function lineClear() {{
            for (let r = ROWS - 1; r >= 0; r--) {{
                if (grid[r].every(value => value !== 0)) {{
                    grid.splice(r, 1);
                    grid.unshift(Array(COLS).fill(0));
                    score += 100;
                    document.getElementById('score').innerText = score;
                    r++;
                }}
            }}
        }}

        function drawMatrix(matrix, offset, color) {{
            matrix.forEach((row, r) => {{
                row.forEach((value, c) => {{
                    if (value) {{
                        context.fillStyle = color || COLORS[value];
                        context.fillRect(offset.x + c, offset.y + r, 1, 1);
                    }}
                }});
            }});
        }}

        function draw() {{
            context.clearRect(0, 0, canvas.width, canvas.height);
            grid.forEach((row, r) => {{
                row.forEach((value, c) => {{
                    if (value) {{
                        context.fillStyle = COLORS[value];
                        context.fillRect(c, r, 1, 1);
                    }}
                }});
            }});
            drawMatrix(currentPiece.shape, {{x: currentPiece.x, y: currentPiece.y}}, COLORS[currentPiece.type]);
        }}

        function drop() {{
            currentPiece.y++;
            if (checkCollision(grid, currentPiece)) {{
                currentPiece.y--;
                merge(grid, currentPiece);
                lineClear();
                resetPiece();
            }}
            draw();
        }}

        setInterval(drop, 400);

        // 신규 포트(8999)가 적용된 웹소켓 연결 연동
        const ws = new WebSocket("{websocket_wss_url}");

        ws.onopen = () => {{
            document.getElementById('ws-status').innerText = "연결 완료 (AI 작전 중)";
            document.getElementById('ws-status').style.color = "#0f0";
            resetPiece();
        }};

        ws.onclose = () => {{
            document.getElementById('ws-status').innerText = "연결 해제됨 (서버 확인 필요)";
            document.getElementById('ws-status').style.color = "red";
        }};

        ws.onmessage = (event) => {{
            const action = JSON.parse(event.data);

            for(let i=0; i < action.targetRot; i++) {{
                currentPiece.shape = currentPiece.shape.map((val, index) => currentPiece.shape.map(row => row[index]).reverse());
            }}
            currentPiece.x = action.targetX;
            draw();
        }};

        function requestAIMove() {{
            if (ws.readyState === WebSocket.OPEN) {{
                const gameState = {{
                    grid: grid.map(row => row.map(v => v ? 1 : 0)),
                    currentPiece: currentPiece.type
                }};
                ws.send(JSON.stringify(gameState));
            }}
        }}
    </script>
</body>
</html>
"""

display(HTML(html_code))
